# Stabilita a konvergence

$$
\partial_T\partial_T \psi = \partial_R\partial_R\psi + \frac{N-1}{R}\partial_R\psi
$$
lze linearizovat jako
$$
\begin{align*}
    \pi &:= -\partial_T \psi,\ \phi := \partial_R \psi\\
    \partial_T \psi &= -\pi \\
    \partial_T \phi &= -\partial_R\pi + \gamma_2 (\partial_R \psi - \phi)\\
    \partial_T \pi &= -\frac{1}{R^{N-1}} \partial_R( R^{N-1}\phi) = -\partial_R\phi - \frac{N-1}{R} \phi
\end{align*}
$$

s vnější okrajovou podmínkou
$$
    \phi - \pi + \frac{N-1}{2R}\psi = 0 .
$$


## Utils

In [ ]:
# !pip install numpy matplotlib

In [ ]:
import numpy as np
from enum import Enum
from dataclasses import dataclass, field
from itertools import pairwise


class Parity(int, Enum):
    Undef = 0
    Odd = 1
    Even = 2


# Vytvoření ekvidistantní mřížky od 0 do rmax s krokem dr
def grid(dr: float, rmax: float = 10.0) -> np.ndarray:
    return np.arange(0.0, rmax + dr, dr)


# Kontrola CFL podmínky stability (c * dt / dr <= 1)
def cfl_check(dt: float, dr: float, c: float = 1.0):
    return c * dt / dr <= 1

### Derivace 4. a 6. řádu

In [ ]:
def deriv_r_o4(arr: np.ndarray, dx: float, parity: int = 0, left_bound_origin: bool = True):
    res = np.zeros_like(arr)
    n = len(arr)

    if n < 5:
        raise ValueError("Pole musí mít alespoň 5 bodů pro 4. řád přesnosti.")

    # Centrální diference
    res[2:-2] = (arr[:-4] - 8 * arr[1:-3] + 8 * arr[3:-1] - arr[4:]) / (12 * dx)

    # Vnější okraj s ošetřením 4. řádu
    res[-1] = (3 * arr[-5] - 16 * arr[-4] + 36 * arr[-3] - 48 * arr[-2] + 25 * arr[-1]) / (12 * dx)
    res[-2] = (-arr[-5] + 6 * arr[-4] - 18 * arr[-3] + 10 * arr[-2] + 3 * arr[-1]) / (12 * dx)

    # Pokud se nejedná o počátek, nastavíme paritu na Nedefinovanou
    applied_parity = parity if left_bound_origin else 0

    match applied_parity:
        # Parity.Odd (lichá parita)
        case 1:
            # Počátek s ošetřením 4. řádu pro lichou funkci
            res[0] = (-arr[2] + 8 * arr[1] - (-1) * 8 * arr[1] + (-1) * arr[2]) / (12 * dx)
            # arr[0] musí být nula s lichou paritou
            # res[1] = (-arr[3] + 8 * arr[2] - 8 * arr[0] + (-1) * arr[1]) / (12 * dx)
            res[1] = (-arr[3] + 8 * arr[2] + (-1) * arr[1]) / (12 * dx)

        # Parity.Even (sudá parita)
        case 2:
            # Počátek s ošetřením 4. řádu pro sudou funkci
            # res[0] je identicky 0 se sudou paritou
            # res[0] = (-arr[2] + 8 * arr[1] - 8 * arr[1] + arr[2]) / (12 * dx)
            res[0] = 0
            res[1] = (-arr[3] + 8 * arr[2] - 8 * arr[0] + arr[1]) / (12 * dx)

        # Parita Nedefinovaná nebo ne v počátku
        case _:
            # Obecné ošetření 5. řádu
            res[0] = (-25 * arr[0] + 48 * arr[1] - 36 * arr[2] + 16 * arr[3] - 3 * arr[4]) / (
                12 * dx
            )
            res[1] = (-3 * arr[0] - 10 * arr[1] + 18 * arr[2] - 6 * arr[3] + arr[4]) / (12 * dx)

    return res


def deriv_r_o6(arr: np.ndarray, dx: float, parity: int = 0, left_bound_origin: bool = True):
    res = np.zeros_like(arr)
    n = len(arr)

    if n < 7:
        raise ValueError("Pole musí mít alespoň 7 bodů pro 6. řád přesnosti.")

    # Pokud se nejedná o počátek, nastavíme paritu na Nedefinovanou
    applied_parity = parity if left_bound_origin else 0
    dx60 = 60 * dx
    # Centrální diference
    res[3:-3] = (
        -arr[:-6] + 9 * arr[1:-5] - 45 * arr[2:-4] + 45 * arr[4:-2] - 9 * arr[5:-1] + arr[6:]
    ) / dx60

    # Vnější okraj: jednostranné 6bodové šablony, přesnost 5. řádu
    res[-1] = (
        -12 * arr[-6] + 75 * arr[-5] - 200 * arr[-4] + 300 * arr[-3] - 300 * arr[-2] + 137 * arr[-1]
    ) / dx60
    res[-2] = (
        3 * arr[-6] - 20 * arr[-5] + 60 * arr[-4] - 120 * arr[-3] + 65 * arr[-2] + 12 * arr[-1]
    ) / dx60
    res[-3] = (
        -2 * arr[-6] + 15 * arr[-5] - 60 * arr[-4] + 20 * arr[-3] + 30 * arr[-2] - 3 * arr[-1]
    ) / dx60

    # Vnitřní okraj: se známou paritou jsou fiktivní body (ghost body) pro r < 0 vyplněny odrazem,
    # takže se použije plná centrální šablona 6. řádu; bez ní se vrátíme k
    # jednostranným šablonám, které jsou pouze 5. řádu.
    match applied_parity:
        # Parity.Odd (lichá parita)
        case 1:
            # Počátek s ošetřením 6. řádu pro lichou funkci (arr[-k] = -arr[k])
            res[0] = 2 * (45 * arr[1] - 9 * arr[2] + arr[3]) / dx60
            res[1] = (arr[2] - 9 * arr[1] + 45 * arr[2] - 9 * arr[3] + arr[4]) / dx60
            res[2] = (arr[1] - 45 * arr[1] + 45 * arr[3] - 9 * arr[4] + arr[5]) / dx60

        # Parity.Even (sudá parita)
        case 2:
            # Počátek s ošetřením 6. řádu pro sudou funkci (arr[-k] = arr[k])
            res[0] = 0
            res[1] = (-arr[2] + 9 * arr[1] - 45 * arr[0] + 45 * arr[2] - 9 * arr[3] + arr[4]) / dx60
            res[2] = (-arr[1] + 9 * arr[0] - 45 * arr[1] + 45 * arr[3] - 9 * arr[4] + arr[5]) / dx60

        # Parita Nedefinovaná nebo ne v počátku
        case _:
            # Obecné ošetření 5. řádu
            res[0] = (
                12 * arr[5]
                - 75 * arr[4]
                + 200 * arr[3]
                - 300 * arr[2]
                + 300 * arr[1]
                - 137 * arr[0]
            ) / dx60
            res[1] = (
                -3 * arr[5] + 20 * arr[4] - 60 * arr[3] + 120 * arr[2] - 65 * arr[1] - 12 * arr[0]
            ) / dx60
            res[2] = (
                2 * arr[5] - 15 * arr[4] + 60 * arr[3] - 20 * arr[2] - 30 * arr[1] + 3 * arr[0]
            ) / dx60

    return res

### RK integrátor

In [ ]:
RK_COEFFICIENT_TABLES = {
    1: ([0], [[0]], [1]),
    2: ([0, 1 / 2], [[0, 0], [1 / 2, 0]], [0, 1]),
    3: ([0, 1 / 2, 1], [[0, 0, 0], [1 / 2, 0, 0], [-1, 2, 0]], [1 / 6, 2 / 3, 1 / 6]),
    4: (
        [0, 1 / 2, 1 / 2, 1],
        [[0, 0, 0, 0], [1 / 2, 0, 0, 0], [0, 1 / 2, 0, 0], [0, 0, 1, 0]],
        [1 / 6, 1 / 3, 1 / 3, 1 / 6],
    ),
    5: (
        [0, 1 / 5, 3 / 10, 4 / 5, 8 / 9, 1, 1],
        [
            [0, 0, 0, 0, 0, 0, 0],
            [1 / 5, 0, 0, 0, 0, 0, 0],
            [3 / 40, 9 / 40, 0, 0, 0, 0, 0],
            [44 / 45, -56 / 15, 32 / 9, 0, 0, 0, 0],
            [19372 / 6561, -25360 / 2187, 64448 / 6561, -212 / 729, 0, 0, 0],
            [9017 / 3168, -355 / 33, 46732 / 5247, 49 / 176, -5103 / 18656, 0, 0],
            [35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84, 0],
        ],
        [35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84, 0],
    ),
    6: (
        [0, 1 / 3, 2 / 3, 1 / 3, 1 / 2, 1 / 2, 1],
        [
            [0, 0, 0, 0, 0, 0, 0],
            [1 / 3, 0, 0, 0, 0, 0, 0],
            [0, 2 / 3, 0, 0, 0, 0, 0],
            [1 / 12, 1 / 3, -1 / 12, 0, 0, 0, 0],
            [-1 / 16, 9 / 8, -3 / 16, -3 / 8, 0, 0, 0],
            [0, 9 / 8, -3 / 8, -3 / 4, 1 / 2, 0, 0],
            [9 / 44, -9 / 11, 63 / 44, 18 / 11, 0, -16 / 11, 0],
        ],
        [11 / 120, 0, 27 / 40, 27 / 40, -4 / 15, -4 / 15, 11 / 120],
    ),
}


class RKp:
    def __init__(self, order: int = 4):
        """
        Args:
            order (int, optional): Řád použité RK metody. Implementované jsou klíče
                RK_COEFFICIENT_TABLES, tj. {1,2,3,4,5,6}. Defaults to 4.
        """

        self.order = order

        # Načtení koeficientů Runge-Kutta
        (c, A, b) = RK_COEFFICIENT_TABLES[order]
        (self.c, self.A, self.b) = (np.array(c), np.array(A), np.array(b))

    # Inicializace hodnot před zahájením procesu integrace
    def Initialize(self, y0: np.ndarray, dydt):
        """
        Args:
            y0 (np.ndarray): Počáteční data. Integrátor jejich vnitřní strukturu neřeší
                (kolik polí je slepeno za sebou je věc `dydt`, ne integrátoru).
            dydt (funkce): Funkce ve tvaru dy/dt = f(t, y)
        """
        self.y = y0.copy()
        self.ICS = y0.copy()
        self.dydt = dydt
        self.t0 = 0.0
        self.t = 0.0
        self.h = 0.0
        self.step_index = 0
        self.y_history = [y0.copy()]
        self.t_history = [0.0]

    # Provedení jednoho kroku metody RKp
    def Integrate(self, t0: float, h: float, tmax: float, output_every: int = 1):
        """
        Odintegruje přesně `round((tmax - t0) / h)` kroků o pevné délce `h`. Časová osa
        je díky tomu ekvidistantní (`t_i = t0 + i * h`) a `tmax` se zaokrouhlí na
        nejbližší násobek `h`. Volání resetuje běh na počáteční data z `Initialize`.

        Args:
            t0 (float): Počáteční čas
            h (float): Časový krok
            tmax (float): Koncový čas integrace
            output_every (int, optional): Do historie se zapisuje každý `output_every`-tý
                krok; počáteční a koncový stav vždy. Defaults to 1.
        """
        if h <= 0:
            raise ValueError("Krok h musí být kladný.")
        if output_every < 1:
            raise ValueError("output_every musí být alespoň 1.")

        self.t0 = t0
        self.t = t0
        self.h = h
        self.step_index = 0
        self.y = self.ICS.copy()
        self.y_history = [self.ICS.copy()]
        self.t_history = [t0]

        n_steps = round((tmax - t0) / h)
        for i in range(1, n_steps + 1):
            self.NextStep(store=(i % output_every == 0 or i == n_steps))

    # Výpočet k pro obecnou Butcherovu tabulku
    def _getKs(self):
        k = np.zeros((len(self.c),) + np.shape(self.y))
        for i in range(len(self.c)):
            t = self.t + self.h * self.c[i]

            dy = np.zeros_like(self.y)
            for j in range(i):
                dy += self.A[i, j] * k[j]
            y = self.y + self.h * dy

            k[i] = self.dydt(t, y)
        return k.copy()

    # Provedení jednoho kroku metody RKp
    def NextStep(self, store: bool = True):
        # Získání hodnot k
        k = self._getKs()
        # Aktualizace hodnot y a t
        self.y += self.h * sum(self.b[i] * k[i] for i in range(len(self.c)))
        self.step_index += 1
        # Čas z indexu kroku, ne akumulací t += h -- ta po desetitisících kroků driftuje
        self.t = self.t0 + self.step_index * self.h
        # Uložení hodnoty y
        if store:
            self.y_history.append(self.y.copy())
            self.t_history.append(self.t)
        return self.step_index

    def GetHistory(self):
        return self.y_history

    def GetTimes(self):
        """Časy odpovídající řádkům `GetHistory()`."""
        return self.t_history

    @classmethod
    def GetAllImplemented(cls):
        return RK_COEFFICIENT_TABLES.keys()

### Dataclass `Run` pro běhy

In [ ]:
@dataclass(frozen=True)
class Run:
    """Odintegrovaný běh rozbalený na jednotlivá pole.

    Integrátor drží stav jako jeden slepený vektor délky 3N; tady je rozřezaný
    na tři pole tvaru (M, N), indexovaná [čas, prostor].

    Attributes:
        r: ekvidistantní mřížka, tvar (N,).
        t: časy uložených snímků, tvar (M,).
        psi, phi, pi: tvar (M, N).
        meta: parametry běhu (r0, sigma, gamma2, dt_integration, ...).
    """

    r: np.ndarray
    t: np.ndarray
    psi: np.ndarray
    phi: np.ndarray
    pi: np.ndarray
    meta: dict = field(default_factory=dict)

    @property
    def dr(self) -> float:
        return float(self.r[1] - self.r[0])

    @property
    def dt(self) -> float:
        """Rozestup uložených snímků = output_every * integrační krok.

        POZOR: není to integrační krok, ten je v meta["dt_integration"].
        """
        return float(self.t[1] - self.t[0])

    @classmethod
    def from_solver(cls, solver, r: np.ndarray, meta: dict | None = None) -> "Run":
        """Rozbalí y(t) tvaru (M, 3N) na psi, phi, pi."""
        y = np.asarray(solver.GetHistory())
        t = np.asarray(solver.GetTimes())
        n = len(r)
        if y.shape != (len(t), 3 * n):
            raise ValueError(f"Historie má tvar {y.shape}, čekal jsem {(len(t), 3 * n)}.")
        return cls(
            r=r, t=t, psi=y[:, :n], phi=y[:, n : 2 * n], pi=y[:, 2 * n :], meta=dict(meta or {})
        )

### Verification utils

In [ ]:
def l2_norm(u: np.ndarray, dx: float) -> float:
    """||u|| = sqrt(dx * sum_j u_j^2) pro jeden časový řez, tvar (N,).

    Na celou historii (M, N) je `l2_norm_history`.

    Faktor dx tam musí být: bez něj norma roste s počtem bodů jako sqrt(N)
    a při zjemňování mřížky by se do všech poměrů zamíchal falešný sqrt(2).
    """
    return float(np.sqrt(dx * np.sum(np.asarray(u) ** 2)))


def l2_norm_history(u: np.ndarray, dx: float) -> np.ndarray:
    """||u(t_i)|| pro každý řez historie: vstup (M, N), výstup (M,)."""
    u = np.asarray(u)
    if u.ndim != 2:
        raise ValueError(f"Čekám historii tvaru (M, N), dostal jsem {u.shape}.")
    return np.sqrt(dx * np.sum(u**2, axis=1))


def orders_from_errors(errors: list[float]) -> list[float]:
    """Z chyb při postupném půlení kroku vrátí řády log2(e_i / e_{i+1}).

    Vrací o jedno číslo méně, než je chyb. Při řádu p mají všechny vyjít p.
    """
    return [float(np.log2(a / b)) for a, b in pairwise(errors)]


def _format_cell(key: str, value) -> str:
    """Text jak je, celá čísla jako celá, řády na 3 desetiny, zbytek vědecky."""
    if isinstance(value, str):
        return value
    if isinstance(value, (int, np.integer)):
        return f"{value:d}"
    if key.startswith(("order", "rad_", "ratio")):
        return f"{value:.3f}" if np.isfinite(value) else "--"
    return f"{value:.4e}"


def print_table(rows: list[dict], title: str = "") -> None:
    """Vytiskne seznam slovníků jako tabulku; šířka sloupců podle obsahu."""
    if title:
        print(f"\n{title}")
    if not rows:
        return
    keys = list(rows[0])
    table = [[_format_cell(k, row[k]) for k in keys] for row in rows]
    widths = [max(len(k), *(len(row[i]) for row in table)) for i, k in enumerate(keys)]
    print("  ".join(k.rjust(w) for k, w in zip(keys, widths, strict=True)))
    for row in table:
        print("  ".join(c.rjust(w) for c, w in zip(row, widths, strict=True)))


# ---------------------------------------------------------------------------
# Test řádu prostorové derivace
# ---------------------------------------------------------------------------


def derivative_order_test(
    func,
    dfunc,
    n_points: list[int],
    r_max: float = 1.0,
    parity: int = 0,
    left_bound_origin: bool = False,
) -> list[dict]:
    """
    Porovná D6 f s analytickou derivací f' na postupně jemnějších mřížkách.

    Args:
        func, dfunc: funkce a její analytická derivace, obě volatelné na numpy poli.
        n_points: počty bodů, každý dvojnásobek předchozího (např. [51, 101, 201]).
        r_max: mřížka je linspace(0, r_max, n).
        parity, left_bound_origin: předá se `deriv_r_o6`. Pro test parity v počátku
            dej parity=Parity.Even (resp. Odd) a left_bound_origin=True.

    Returns:
        Řádky s dx a maximem |chyby| zvlášť ve třech oblastech: `vnitrek`
        (centrální stencil), `pocatek` (3 body u r = 0) a `okraj` (3 body u r_max),
        plus naměřené řády `rad_*`.

    Proč zvlášť: chyba není po mřížce rovnoměrná a maximum přes celou mřížku by
    vždycky změřilo jen ten nejhorší kus. Očekávané řády jsou

        vnitrek  6   (centrální stencil)
        pocatek  6   s paritou (ghost pointy dodá zrcadlení), jinak 5
        okraj    5   (jednostranné stencily, tam parita nepomůže)

    Pozor na strojovou přesnost: pro dx pod ~0.005 padá chyba 6. řádu k 1e-13
    a naměřený řád začne skákat. To není chyba schématu, to je podlaha double.
    """
    rows = []
    prev = {}
    for n in n_points:
        r, dx = np.linspace(0.0, r_max, n, retstep=True)
        err = np.abs(
            deriv_r_o6(func(r), dx, parity=parity, left_bound_origin=left_bound_origin) - dfunc(r)
        )
        now = {
            "vnitrek": float(np.max(err[3:-3])),
            "pocatek": float(np.max(err[:3])),
            "okraj": float(np.max(err[-3:])),
        }
        row = {"n": n, "dx": dx, **now}
        for key, value in now.items():
            old = prev.get(key)
            row[f"rad_{key}"] = np.log2(old / value) if old and value else np.nan
        rows.append(row)
        prev = now
    return rows


# ---------------------------------------------------------------------------
# Test self-konvergence
# ---------------------------------------------------------------------------


def self_convergence(
    coarse: np.ndarray,
    medium: np.ndarray,
    fine: np.ndarray,
    dx_coarse: float,
    space_factor: int = 2,
) -> tuple[float, float, float]:
    """
    Řád z trojice řešení v jednom čase, každé s 2x menším krokem.

    Args:
        coarse, medium, fine: 1D řezy ve stejném čase, každý s 2x menším krokem.
        dx_coarse: prostorový krok nejhrubšího řezu (jen pro škálování normy).
        space_factor: 2 při zjemňování PROSTORU -- pole mají N, 2N-1, 4N-3 bodů
            a podvzorkují se na nejhrubší mřížku. 1 při zjemňování ČASU, kde
            se mřížka nemění a všechna tři pole mají stejnou délku.

    Returns:
        (řád, ||u_h - u_h/2||, ||u_h/2 - u_h/4||)
    """
    if space_factor < 1:
        raise ValueError("space_factor musí být alespoň 1.")
    m = medium[::space_factor]
    f = fine[:: space_factor**2]
    if not (len(coarse) == len(m) == len(f)):
        raise ValueError(
            f"Nesedí délky: po podvzorkování {len(coarse)}, {len(m)}, {len(f)} bodů. "
            f"Se space_factor={space_factor} čekám vstupy délky N, "
            f"{space_factor}N-{space_factor - 1}, {space_factor**2}N-{space_factor**2 - 1}."
        )
    d1 = l2_norm(coarse - m, dx_coarse)
    d2 = l2_norm(m - f, dx_coarse)
    order = float(np.log2(d1 / d2)) if d2 > 0 else np.nan
    return order, d1, d2


def self_convergence_history(
    coarse: np.ndarray,
    medium: np.ndarray,
    fine: np.ndarray,
    dx_coarse: float,
    space_factor: int = 2,
) -> np.ndarray:
    """
    Totéž jako `self_convergence`, ale pro celou historii -- vrátí řád Q(t).

    Vstupy mají tvar (M, N), (M, 2N-1), (M, 4N-3); časové osy musí být shodné,
    tj. `output_every` musí škálovat s krokem, aby snímky padly na stejné časy.
    V t = 0 je rozdíl nulový, takže Q(0) je nan -- to je v pořádku, počáteční
    data jsou na společných bodech identická.
    """
    m = medium[:, ::space_factor]
    d1 = l2_norm_history(coarse - m, dx_coarse)
    d2 = l2_norm_history(m - fine[:, :: space_factor**2], dx_coarse)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(d2 > 0, np.log2(d1 / d2), np.nan)


# ---------------------------------------------------------------------------
# Test porušení constraintu
# ---------------------------------------------------------------------------


def constraint_violation(
    psi: np.ndarray, phi: np.ndarray, dx: float, parity: int = Parity.Even
) -> np.ndarray:
    """
    C = D6 psi - phi, vstup i výstup tvaru (M, N)
    """
    d_psi = np.array([deriv_r_o6(row, dx, parity=parity) for row in np.atleast_2d(psi)])
    return d_psi - np.atleast_2d(phi)


def damping_prediction(t: np.ndarray, c0: float, gamma2: float) -> np.ndarray:
    """||C(0)|| exp(-gamma2 t)."""
    return c0 * np.exp(-gamma2 * np.asarray(t))


## N-dimenzionální systém

In [ ]:
def ndim_lin_wave_equation_div_rhs(
    r_coordinate: np.ndarray, dimension_n: int, dx: float, gamma2: float, N_of_points: int
):
    r_to_n_minus_1 = r_coordinate ** (dimension_n - 1)
    phi_times_r_to_n_minus_1_parity = Parity.Odd if dimension_n % 2 == 1 else Parity.Even

    def _system(t: float, y: np.ndarray):
        """
        d/dt[psi] = - pi
        d/dt[phi] = - d/dr[pi] + gamma2 (d/dr[psi] - phi)
        d/dt[pi]  = - 1/r^(n-1) d/dr[r^(n-1) phi] === divergencni tvar
        + Sommerfeldova odchozí podmínka na pravém okraji
        """
        assert len(y) % 3 == 0, (
            "Očekává se, že vstup systému bude tvořen třemi stejně dlouhými funkcemi"
        )

        # 1. Rozbalení vektoru y na jednotlivá pole
        psi = y[0:N_of_points]
        phi = y[N_of_points : 2 * N_of_points]
        pi = y[2 * N_of_points : 3 * N_of_points]

        # 2. Inicializace derivací (rhs)
        dpsi_dt = np.zeros(N_of_points)
        dphi_dt = np.zeros(N_of_points)
        dpi_dt = np.zeros(N_of_points)

        # --- d/dt[psi] = - pi ---
        dpsi_dt = -pi

        # --- d/dt[phi] = - d/dr[pi] + gamma2 (d/dr[psi] - phi)  - ---
        dphi_dt[1:] = (
            -deriv_r_o6(pi, dx, parity=Parity.Even)[1:]
            + gamma2 * (deriv_r_o6(psi, dx, parity=Parity.Even) - phi)[1:]
        )
        # Sudost Psi v r zaručuje v počátku Phi (= dPsi/dr) = 0
        dphi_dt[0] = 0.0

        # --- d/dt[pi] = - 1/r^(n-1) d/dr[r^(n-1) phi] ---
        dpi_dt[1:] = (
            -deriv_r_o6(phi * r_to_n_minus_1, dx, parity=phi_times_r_to_n_minus_1_parity)[1:]
            / r_to_n_minus_1[1:]
        )
        dpi_dt[0] = -dimension_n * deriv_r_o6(phi, dx, parity=Parity.Odd)[0]

        # --- Sommerfeld (odchozí vlna) ---
        # d_T psi + d_R psi + psi/R = 0, tj. pi = phi + psi/R;
        # derivací podle času: d_T pi = d_T phi + (d_T psi)/R = d_T phi - pi/R
        last_dphi_dt = dphi_dt[-1]
        last_dpi_dt = dpi_dt[-1]

        if dimension_n == 1:
            dpi_dt[-1] = last_dphi_dt
            dphi_dt[-1] = last_dpi_dt
        else:
            factor = pi[-1] * (dimension_n - 1) / (2 * r_coordinate[-1])
            dpi_dt[-1] = last_dphi_dt - factor
            dphi_dt[-1] = last_dpi_dt + factor

        # # --- Dirichlet ---
        # dpi_dt[-1] = 0.0
        # dphi_dt[-1] = 0.0

        # 3. Zabalení zpět do jednoho vektoru
        return np.concatenate([dpsi_dt, dphi_dt, dpi_dt])

    return _system


def ndim_lin_wave_equation_rhs(
    r_coordinate: np.ndarray, dimension_n: int, dx: float, gamma2: float, N_of_points: int
):
    def _system(t: float, y: np.ndarray):
        """
        d/dt[psi] = - pi
        d/dt[phi] = - d/dr[pi] + gamma2 (d/dr[psi] - phi)
        d/dt[pi]  = - d/dr[phi] - (n-1)/r phi
        + Sommerfeldova odchozí podmínka na pravém okraji
        """
        assert len(y) % 3 == 0, (
            "Očekává se, že vstup systému bude tvořen třemi stejně dlouhými funkcemi"
        )

        # 1. Rozbalení vektoru y na jednotlivá pole
        psi = y[0:N_of_points]
        phi = y[N_of_points : 2 * N_of_points]
        pi = y[2 * N_of_points : 3 * N_of_points]

        # 2. Inicializace derivací (rhs)
        dpsi_dt = np.zeros(N_of_points)
        dphi_dt = np.zeros(N_of_points)
        dpi_dt = np.zeros(N_of_points)

        # --- d/dt[psi] = - pi ---
        dpsi_dt = -pi

        # --- d/dt[phi] = - d/dr[pi] + gamma2 (d/dr[psi] - phi)  - ---
        dphi_dt[1:] = (
            -deriv_r_o6(pi, dx, parity=Parity.Even)[1:]
            + gamma2 * (deriv_r_o6(psi, dx, parity=Parity.Even) - phi)[1:]
        )
        # Sudost Psi v r zaručuje v počátku Phi (= dPsi/dr) = 0
        dphi_dt[0] = 0.0

        # --- d/dt[pi] = - d/dr[phi] - (n-1)/r phi---
        dpi_dt[1:] = (
            -deriv_r_o6(phi, dx, parity=Parity.Odd)[1:]
            - phi[1:] * (dimension_n - 1) / r_coordinate[1:]
        )
        dpi_dt[0] = -dimension_n * deriv_r_o6(phi, dx, parity=Parity.Odd)[0]

        # --- Sommerfeld (odchozí vlna) ---
        # d_T psi + d_R psi + psi/R = 0, tj. pi = phi + psi/R;
        # derivací podle času: d_T pi = d_T phi + (d_T psi)/R = d_T phi - pi/R
        last_dphi_dt = dphi_dt[-1]
        last_dpi_dt = dpi_dt[-1]

        if dimension_n == 1:
            dpi_dt[-1] = last_dphi_dt
            dphi_dt[-1] = last_dpi_dt
        else:
            factor = pi[-1] * (dimension_n - 1) / (2 * r_coordinate[-1])
            dpi_dt[-1] = last_dphi_dt - factor
            dphi_dt[-1] = last_dpi_dt + factor

        # # --- Dirichlet ---
        # dpi_dt[-1] = 0.0
        # dphi_dt[-1] = 0.0

        # 3. Zabalení zpět do jednoho vektoru
        return np.concatenate([dpsi_dt, dphi_dt, dpi_dt])

    return _system

In [ ]:
def exit_time(run: Run) -> float:
    """t = rmax - r0, tj. kdy střed pulsu dorazí na vnější okraj."""
    return float(run.r[-1] - run.meta["r0"])

## Případ 1D

### Utils

In [ ]:
# `exact_psi` níž platí jen pro n = 1
DIMENSION = 1


def initial_data_1d(
    r: np.ndarray, r0: float = 4.0, sigma: float = 0.5, phi_source: str = "discrete"
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """psi = exp(-(r - r0)^2 / sigma^2), phi = pi = d/dr psi.

    Volba pi = phi = d/dr psi znamená psi_t = -pi = -psi_r, tedy čistě pravoběžnou vlnu.

    `phi_source` určuje, jak se derivace spočítá, a tím i počáteční hodnotu
    constraintu C = D6 psi - phi:

      * "discrete" -- operátorem D6, tedy stejným jako evoluce: C(0) = 0 na strojovou
        přesnost.
      * "analytic" -- d/dr psi = -2 (r - r0) / sigma^2 * psi, tedy C(0) = -tau, kde tau
        je lokální truncation error D6, čili O(dr^6) ve vnitřku a O(dr^5) na okrajových
        řádcích. Tlumení C(t) = C(0) exp(-gamma2 t)
        se dá testovat jen s C(0) != 0.
    """
    psi = np.exp(-np.power(r - r0, 2) / np.power(sigma, 2))
    if phi_source == "discrete":
        # Parita musí odpovídat evoluci i `constraint_violation`, jinak C(0) != 0.
        d_psi = deriv_r_o6(psi, r[1] - r[0], parity=Parity.Even)
    elif phi_source == "analytic":
        d_psi = -2.0 * (r - r0) / np.power(sigma, 2) * psi
    else:
        raise ValueError(f"Neznámý phi_source: {phi_source!r}")
    return psi, d_psi, d_psi.copy()


def run_1d_gaussian(
    dr: float,
    dt: float,
    tmax: float,
    *,
    rmax: float = 10.0,
    r0: float = 4.0,
    sigma: float = 0.5,
    gamma2: float = 0.1,
    rk_order: int = 6,
    output_every: int = 1,
    phi_source: str = "discrete",
) -> Run:
    """Odintegruje jeden běh a vrátí ho jako `Run`.

    Args:
        dr, dt: prostorový a časový krok.
        tmax: koncový čas, zaokrouhlí se na nejbližší násobek dt.
        rmax: vnější okraj domény.
        r0, sigma: střed a šířka počátečního gaussiánu.
        gamma2: koeficient tlumení constraintu.
        rk_order: řád RK; default 6, aby časová chyba nedominovala nad prostorovou
            derivací 6. řádu.
        output_every: ukládá se každý output_every-tý krok. Při porovnávání rozlišení
            ho škáluj s dt, aby všechny běhy měly stejné výstupní časy.
        phi_source: viz `initial_data`.
    """
    r = grid(dr, rmax)
    n_points = len(r)
    psi, phi, pi = initial_data_1d(r, r0, sigma, phi_source=phi_source)

    solver = RKp(order=rk_order)
    solver.Initialize(
        np.concatenate([psi, phi, pi]),
        ndim_lin_wave_equation_div_rhs(
            r_coordinate=r,
            dimension_n=DIMENSION,
            dx=dr,
            gamma2=gamma2,
            N_of_points=n_points,
        ),
    )
    solver.Integrate(t0=0.0, h=dt, tmax=tmax, output_every=output_every)

    return Run.from_solver(
        solver,
        r=r,
        meta={
            "r0": r0,
            "sigma": sigma,
            "gamma2": gamma2,
            "rk_order": rk_order,
            "dt_integration": dt,
            "dimension_n": DIMENSION,
            "phi_source": phi_source,
        },
    )


def exact_psi_1d(run: Run) -> np.ndarray:
    """psi(t, r) = exp(-(r - r0 - t)^2 / sigma^2) na mřížce běhu, tvar (M, N).

    Řešení počáteční úlohy na celé přímce. V doméně [0, rmax] platí, dokud puls
    nedorazí na vnější okraj, tj. do t ~ rmax - r0.
    """
    r0, sigma = run.meta["r0"], run.meta["sigma"]
    shifted = run.r[np.newaxis, :] - r0 - run.t[:, np.newaxis]
    return np.exp(-np.power(shifted, 2) / np.power(sigma, 2))


### Analýza

In [ ]:
import matplotlib.pyplot as plt

# --- Parametry ---
DR_1D = 0.1
DT_1D = 0.005
T_MAX_1D = 30.0
OUTPUT_EVERY_1D = 10
GAMMA2_1D = 0.1

assert cfl_check(DT_1D, DR_1D), "CFL violated"

run_1d = run_1d_gaussian(
    DR_1D, DT_1D, T_MAX_1D, output_every=OUTPUT_EVERY_1D, gamma2=GAMMA2_1D, rk_order=4
)

# --- Chyba vůči exaktnímu řešení ---
# Exaktní řešení je pravoběžný gaussián na celé přímce, takže v naší domény platí
# jen dokud puls nedorazí na vnější okraj -- to je exit_time(run_1d).
error_1d = run_1d.psi - exact_psi_1d(run_1d)
error_l2_1d = l2_norm_history(error_1d, run_1d.dr)

# --- Porušení constraintu C = D6 psi - phi ---
# S phi_source="discrete" (default) je C(0) = 0 identicky, takže všechno naměřené
# vzniklo z okrajových podmínek. Na test tlumení C(t) = C(0) exp(-gamma2 t) je
# potřeba běh s C(0) != 0, tj. phi_source="analytic" -- viz verify.py, test C.
constraint_l2_1d = l2_norm_history(
    constraint_violation(run_1d.psi, run_1d.phi, run_1d.dr), run_1d.dr
)

t_exit_1d = exit_time(run_1d)
before_1d = run_1d.t <= t_exit_1d

print(f"exit_time         = {t_exit_1d:.2f}")
print(f"||e||_2 max do    = {error_l2_1d[before_1d].max():.3e}")
print(f"||C||_2           = {constraint_l2_1d[0]:.3e} -> {constraint_l2_1d[before_1d][-1]:.3e}")
print(f"||e||_2 max po    = {error_l2_1d[~before_1d].max():.3e}  (odraz od okraje, ne chyba)")
# Za exit_time je exaktní řešení v doméně nulové, takže ||e|| = ||psi|| a měří jen
# to, co po pulsu zbylo -- odraz od Sommerfeldovy podmínky. Konverguje k nule
# s dr^5 (řád vnějšího okraje), takže při dr <= 0.05 je pod zaokrouhlením.

# --- Grafy ---
fig_1d, (ax1_1d, ax2_1d) = plt.subplots(1, 2, figsize=(13, 5))

ax1_1d.semilogy(run_1d.t, error_l2_1d, lw=1.5, label=r"$\|e\|_2$")
ax1_1d.semilogy(run_1d.t, constraint_l2_1d, lw=1.5, label=r"$\|C\|_2$")
ax1_1d.axvline(exit_time(run_1d), color="black", lw=1, ls=":", label="odchod pulsu")
ax1_1d.set_xlabel("t")
ax1_1d.set_title(f"L2 normy, dr = {run_1d.dr:g}")
ax1_1d.grid(True, ls="--", alpha=0.6)
ax1_1d.legend()

scale_1d = abs(error_1d).max() or 1.0
mesh_1d = ax2_1d.pcolormesh(
    run_1d.r, run_1d.t, error_1d, cmap="RdBu_r", vmin=-scale_1d, vmax=scale_1d, shading="nearest"
)
fig_1d.colorbar(mesh_1d, ax=ax2_1d, label="e")
ax2_1d.set_xlabel("r")
ax2_1d.set_ylabel("t")
ax2_1d.set_title(f"e(t, r), dr = {run_1d.dr:g}")

fig_1d.tight_layout()
plt.show()

### Řád konvergence

In [ ]:
T_MAX_1D = 4.0  # bezpečně pod exit_time = rmax - r0 = 6
GAMMA2_1D = 0.1
RK_ORDER_1D = 4

# ---------------------------------------------------------------------------
# Sken 1: prostorový řád proti exaktnímu řešení
# ---------------------------------------------------------------------------

DT_FIXED_1D = 0.0025  # tak malé, aby časová chyba RK4 byla zanedbatelná
DR_FIXED_1D_SCAN1 = 0.1  # Placeholder for the first scan's DR, if needed for CFL check
assert cfl_check(DT_FIXED_1D, DR_FIXED_1D_SCAN1), "CFL violated for Scan 1 (dt_fixed_1d, first dr)"

rows = []
prev = None
for dr in [0.1, 0.05, 0.025]:
    run_1d = run_1d_gaussian(
        dr, DT_FIXED_1D, T_MAX_1D, output_every=1, gamma2=GAMMA2_1D, rk_order=RK_ORDER_1D
    )
    err = l2_norm(run_1d.psi[-1] - exact_psi_1d(run_1d)[-1], run_1d.dr)
    rows.append(
        {
            "dr": dr,
            "err_l2": err,
            "ratio": prev / err if prev else np.nan,
            "order": np.log2(prev / err) if prev else np.nan,
        }
    )
    prev = err

print_table(rows, f"Sken 1 -- prostorový řád (dt_1d = {DT_FIXED_1D}, t_1d = {T_MAX_1D})   čekám 6")

# ---------------------------------------------------------------------------
# Sken 2: časový řád self-konvergencí
# ---------------------------------------------------------------------------

DR_FIXED_1D = 0.05
DT_COARSE_1D = 0.02  # dt/dr = 0.4, CFL v pohodě
assert cfl_check(DT_COARSE_1D, DR_FIXED_1D), "CFL violated for Scan 2 (dt_coarse_1d, dr_fixed_1d)"

psis_1d = []
for k in range(3):
    # output_every se zdvojuje s půlením dt -> snímky padnou na stejné časy
    run_1d = run_1d_gaussian(
        DR_FIXED_1D,
        DT_COARSE_1D / 2**k,
        T_MAX_1D,
        output_every=2**k,
        gamma2=GAMMA2_1D,
        rk_order=RK_ORDER_1D,
    )
    psis_1d.append(run_1d.psi[-1])

# space_factor=1: mřížka se nemění, zjemňuje se jen čas
order, d1, d2 = self_convergence(psis_1d[0], psis_1d[1], psis_1d[2], DR_FIXED_1D, space_factor=1)
print(
    f"\nSken 2 -- časový řád self-konvergencí (dr_1d = {DR_FIXED_1D}, t_1d = {T_MAX_1D})   čekám 4"
)
print(f"  ||u_dt - u_dt/2||   = {d1:.4e}")
print(f"  ||u_dt/2 - u_dt/4|| = {d2:.4e}")
print(f"  poměr               = {d1 / d2:.2f}   (má být 2^4 = 16)")
print(f"  řád                 = {order:.3f}")

## 3D případ, divergenční tvar

### Utils

In [ ]:
DIMENSION_3 = 3


def initial_data_3d(
    r: np.ndarray, r0: float, sigma: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Odchozí gaussovský puls: psi = exp(-((r-r0)/sigma)^2), pi = phi + a*psi/r.

    Odchozí řešení v n rozměrech má tvar psi = U(r-t) / r^a s a = (n-1)/2. Z toho

        pi  = -d/dt psi = U'(r-t) / r^a,
        phi =  d/dr psi = U'(r-t) / r^a - a U(r-t) / r^(a+1) = pi - a psi/r,

    tedy pi = phi + a psi/r. To je zároveň Sommerfeldova podmínka, kterou používá
    `we_systems` na vnějším okraji.

    phi se počítá diskrétním operátorem (ne analyticky), takže počáteční porušení
    constraintu C = D6 psi - phi je nulové na strojovou přesnost.
    """
    psi = np.exp(-np.power((r - r0) / sigma, 2))
    phi = deriv_r_o6(psi, r[1] - r[0], parity=Parity.Even)

    a = (DIMENSION_3 - 1) / 2
    pi = phi.copy()
    if a:
        # V r = 0 je psi(0) ~ exp(-(r0/sigma)^2), pro r0=4, sigma=0.5 tedy e^-64.
        # Limita a*psi/r je tam fakticky nula, takže pi[0] = phi[0].
        pi[1:] = phi[1:] + a * psi[1:] / r[1:]
    return psi, phi, pi


def run_3d_gaussian_div(
    dr: float,
    dt: float,
    tmax: float,
    *,
    rmax: float = 10.0,
    r0: float = 4.0,
    sigma: float = 0.5,
    gamma2: float = 0.1,
    rk_order: int = 4,
    output_every: int = 1,
) -> Run:
    """Odintegruje jeden běh a vrátí ho jako `Run`.

    Args:
        dr, dt: prostorový a časový krok.
        tmax: koncový čas, zaokrouhlí se na nejbližší násobek dt.
        dimension_n: počet prostorových rozměrů (3 = sférická symetrie).
        rhs_builder: `ndim_lin_wave_equation_rhs` (nedivergenční) nebo
            `ndim_lin_wave_equation_div_rhs` (divergenční) z `we_systems`.
        output_every: ukládá se každý output_every-tý krok. Při porovnávání
            rozlišení ho škáluj s dt, aby běhy měly stejné výstupní časy.
    """
    r = grid(dr, rmax)
    n_points = len(r)
    psi, phi, pi = initial_data_3d(r, r0, sigma)

    solver = RKp(order=rk_order)
    solver.Initialize(
        np.concatenate([psi, phi, pi]),
        ndim_lin_wave_equation_div_rhs(
            r_coordinate=r,
            dimension_n=DIMENSION_3,
            dx=dr,
            gamma2=gamma2,
            N_of_points=n_points,
        ),
    )
    solver.Integrate(t0=0.0, h=dt, tmax=tmax, output_every=output_every)

    return Run.from_solver(
        solver,
        r=r,
        meta={
            "r0": r0,
            "sigma": sigma,
            "gamma2": gamma2,
            "rk_order": rk_order,
            "dt_integration": dt,
            "dimension_n": DIMENSION_3,
            "system": ndim_lin_wave_equation_div_rhs.__name__,
        },
    )


def exact_psi_3d(run: Run) -> np.ndarray:
    """Odchozí řešení psi(t, r) = U(r - t) / r^a, a = (n-1)/2, tvar (M, N).

    U(x) = x^a exp(-((x - r0)/sigma)^2), takže v t = 0 vyjde psi = exp(-((r-r0)/sigma)^2)
    pro každé n.

    Platí jen pro n = 1 a n = 3 -- jen tam má radiální vlnová rovnice d'Alembertovo
    řešení tohoto tvaru (pro sudé n Huygensův princip neplatí).

    Dvě omezení, stejná jako v 1D:
      * platí, dokud puls nedorazí na vnější okraj, tj. do t = rmax - r0 (`exit_time`),
      * předpokládá, že levý chvost je pod strojovou přesností, aby řešení
        neodporovalo regularitě v počátku. Pro r0=4, sigma=0.5 je exp(-64) ~ 1e-28.
    """
    n = run.meta["dimension_n"]
    if n not in (1, 3):
        raise ValueError(f"Exaktní řešení mám jen pro n = 1 a n = 3, dostal jsem n = {n}.")

    r0, sigma = run.meta["r0"], run.meta["sigma"]
    x = run.r[np.newaxis, :] - run.t[:, np.newaxis]
    gauss = np.exp(-np.power((x - r0) / sigma, 2))
    if n == 1:
        return gauss

    # n = 3: psi = (r - t) * gauss / r. V r = 0 je limita nulová -- puls tam nedojde.
    out = np.zeros_like(gauss)
    out[:, 1:] = x[:, 1:] * gauss[:, 1:] / run.r[np.newaxis, 1:]
    return out

### Analýza

In [ ]:
import matplotlib.pyplot as plt

# --- Parametry pro divergenční tvar ---
DR_3D_div = 0.1
DT_3D_div = 0.0005
T_MAX_3D_div = 8.0
OUTPUT_EVERY_3D_div = 10
GAMMA2_3D_div = 0.1

assert cfl_check(DT_3D_div, DR_3D_div), "CFL violated for divergent form"

run_3d_div = run_3d_gaussian_div(
    DR_3D_div,
    DT_3D_div,
    T_MAX_3D_div,
    output_every=OUTPUT_EVERY_3D_div,
    gamma2=GAMMA2_3D_div,
    rk_order=4,
)

# --- Chyba vůči exaktnímu řešení (divergenční tvar) ---
error_3d_div = run_3d_div.psi - exact_psi_3d(run_3d_div)
error_l2_3d_div = l2_norm_history(error_3d_div, run_3d_div.dr)

# --- Porušení constraintu C = D6 psi - phi (divergenční tvar) ---
constraint_l2_3d_div = l2_norm_history(
    constraint_violation(run_3d_div.psi, run_3d_div.phi, run_3d_div.dr), run_3d_div.dr
)

t_exit_3d_div = exit_time(run_3d_div)
before_3d_div = run_3d_div.t <= t_exit_3d_div

print("-- Divergentní tvar ---")
print(f"exit_time         = {t_exit_3d_div:.2f}")
print(f"||e||_2 max do    = {error_l2_3d_div[before_3d_div].max():.3e}")
print(
    f"||C||_2           = {constraint_l2_3d_div[0]:.3e} -> {constraint_l2_3d_div[before_3d_div][-1]:.3e}"
)
print(
    f"||e||_2 max po    = {error_l2_3d_div[~before_3d_div].max():.3e}  (odraz od okraje, ne chyba)"
)
print()

# --- Parametry pro nedivergenční tvar ---
DR_3D = 0.1
DT_3D = 0.005
T_MAX_3D = 8.0
OUTPUT_EVERY_3D = 10
GAMMA2_3D = 0.1

assert cfl_check(DT_3D, DR_3D), "CFL violated for non-divergent form"

run_3d = run_3d_gaussian_div(
    DR_3D, DT_3D, T_MAX_3D, output_every=OUTPUT_EVERY_3D, gamma2=GAMMA2_3D, rk_order=4
)

# --- Chyba vůči exaktnímu řešení (nedivergenční tvar) ---
error_3d = run_3d.psi - exact_psi_3d(run_3d)
error_l2_3d = l2_norm_history(error_3d, run_3d.dr)

# --- Porušení constraintu C = D6 psi - phi (nedivergenční tvar) ---
constraint_l2_3d = l2_norm_history(
    constraint_violation(run_3d.psi, run_3d.phi, run_3d.dr), run_3d.dr
)

t_exit_3d = exit_time(run_3d)
before_3d = run_3d.t <= t_exit_3d

print("-- Nedivergentní tvar ---")
print(f"exit_time         = {t_exit_3d:.2f}")
print(f"||e||_2 max do    = {error_l2_3d[before_3d].max():.3e}")
print(f"||C||_2           = {constraint_l2_3d[0]:.3e} -> {constraint_l2_3d[before_3d][-1]:.3e}")
print(f"||e||_2 max po    = {error_l2_3d[~before_3d].max():.3e}  (odraz od okraje, ne chyba)")
print()

# --- Grafy ---
fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)

# Divergentní tvar - L2 normy
ax1_3d_div = axes[0, 0]
ax1_3d_div.semilogy(run_3d_div.t, error_l2_3d_div, lw=1.5, label=r"$\|e\|_2$")
ax1_3d_div.semilogy(run_3d_div.t, constraint_l2_3d_div, lw=1.5, label=r"$\|C\|_2$")
ax1_3d_div.axvline(exit_time(run_3d_div), color="black", lw=1, ls=":", label="odchod pulsu")
ax1_3d_div.set_xlabel("t")
ax1_3d_div.set_title(f"L2 normy (divergentní), dr = {run_3d_div.dr:g}")
ax1_3d_div.grid(True, ls="--", alpha=0.6)
ax1_3d_div.legend()

# Divergentní tvar - e(t, r)
ax2_3d_div = axes[0, 1]
scale_3d_div = abs(error_3d_div).max() or 1.0
mesh_3d_div = ax2_3d_div.pcolormesh(
    run_3d_div.r,
    run_3d_div.t,
    error_3d_div,
    cmap="RdBu_r",
    vmin=-scale_3d_div,
    vmax=scale_3d_div,
    shading="nearest",
)
fig.colorbar(mesh_3d_div, ax=ax2_3d_div, label="e")
ax2_3d_div.set_xlabel("r")
ax2_3d_div.set_ylabel("t")
ax2_3d_div.set_title(f"e(t, r) (divergentní), dr = {run_3d_div.dr:g}")

# Nedivergentní tvar - L2 normy
ax1_3d = axes[1, 0]
ax1_3d.semilogy(run_3d.t, error_l2_3d, lw=1.5, label=r"$\|e\|_2$")
ax1_3d.semilogy(run_3d.t, constraint_l2_3d, lw=1.5, label=r"$\|C\|_2$")
ax1_3d.axvline(exit_time(run_3d), color="black", lw=1, ls=":", label="odchod pulsu")
ax1_3d.set_xlabel("t")
ax1_3d.set_title(f"L2 normy (nedivergentní), dr = {run_3d.dr:g}")
ax1_3d.grid(True, ls="--", alpha=0.6)
ax1_3d.legend()

# Nedivergentní tvar - e(t, r)
ax2_3d = axes[1, 1]
scale_3d = abs(error_3d).max() or 1.0
mesh_3d = ax2_3d.pcolormesh(
    run_3d.r, run_3d.t, error_3d, cmap="RdBu_r", vmin=-scale_3d, vmax=scale_3d, shading="nearest"
)
fig.colorbar(mesh_3d, ax=ax2_3d, label="e")
ax2_3d.set_xlabel("r")
ax2_3d.set_ylabel("t")
ax2_3d.set_title(f"e(t, r) (nedivergentní), dr = {run_3d.dr:g}")

plt.show()

### Řád konvergence

In [ ]:
T_MAX_div = 4.0
GAMMA2_div = 0.1
RK_ORDER_div = 4

# ---------------------------------------------------------------------------
# Sken 1: prostorový řád proti exaktnímu řešení
# ---------------------------------------------------------------------------

DT_FIXED_div = 0.0025  # tak malé, aby časová chyba RK4 byla zanedbatelná
DR_FIXED_DIV_SCAN1 = 0.1  # Placeholder for the first scan's DR, if needed for CFL check
assert cfl_check(DT_FIXED_div, DR_FIXED_DIV_SCAN1), (
    "CFL violated for Scan 1 (dt_fixed_div, first dr)"
)

rows = []
prev = None
for dr in [0.1, 0.05, 0.025]:
    run_div = run_3d_gaussian_div(
        dr, DT_FIXED_div, T_MAX_div, output_every=1, gamma2=GAMMA2_div, rk_order=RK_ORDER_div
    )
    err = l2_norm(run_div.psi[-1] - exact_psi_3d(run_div)[-1], run_div.dr)
    rows.append(
        {
            "dr": dr,
            "err_l2": err,
            "ratio": prev / err if prev else np.nan,
            "order": np.log2(prev / err) if prev else np.nan,
        }
    )
    prev = err

print_table(
    rows, f"Sken 1 -- prostorový řád (dt_div = {DT_FIXED_div}, t_div = {T_MAX_div})   čekám 6"
)

# ---------------------------------------------------------------------------
# Sken 2: časový řád self-konvergencí
# ---------------------------------------------------------------------------

DR_FIXED_div = 0.05
DT_COARSE_div = 0.02  # dt/dr = 0.4, CFL v pohodě
assert cfl_check(DT_COARSE_div, DR_FIXED_div), (
    "CFL violated for Scan 2 (dt_coarse_div, dr_fixed_div)"
)

psis_div = []
for k in range(3):
    # output_every se zdvojuje s půlením dt -> snímky padnou na stejné časy
    run_div = run_3d_gaussian_div(
        DR_FIXED_div,
        DT_COARSE_div / 2**k,
        T_MAX_div,
        output_every=2**k,
        gamma2=GAMMA2_div,
        rk_order=RK_ORDER_div,
    )
    psis_div.append(run_div.psi[-1])

# space_factor=1: mřížka se nemění, zjemňuje se jen čas
order, d1, d2 = self_convergence(
    psis_div[0], psis_div[1], psis_div[2], DR_FIXED_div, space_factor=1
)
print(
    f"\nSken 2 -- časový řád self-konvergencí (dr_div = {DR_FIXED_div}, t_div = {T_MAX_div})   čekám 4"
)
print(f"  ||u_dt - u_dt/2||   = {d1:.4e}")
print(f"  ||u_dt/2 - u_dt/4|| = {d2:.4e}")
print(f"  poměr               = {d1 / d2:.2f}   (má být 2^4 = 16)")
print(f"  řád                 = {order:.3f}")

### Iterace přes dR

In [ ]:
import matplotlib.pyplot as plt

# --- Parametry ---
DT_3D_div = 0.0005
T_MAX_3D_div = 8.0
OUTPUT_EVERY_3D_div = 10
GAMMA2_3D_div = 0.1

DR_values = [0.1, 0.05, 0.01, 0.005]
num_dr_values = len(DR_values)

# Create a single figure with multiple subplots
# Each DR will occupy one row with two subplots (L2 norms and e(t,r))
fig_all, axes_all = plt.subplots(num_dr_values, 2, figsize=(13, 5 * num_dr_values), squeeze=False)

for i, DR_3D_div in enumerate(DR_values):
    assert cfl_check(DT_3D_div, DR_3D_div), "CFL violated"

    run_3d_div = run_3d_gaussian_div(
        DR_3D_div,
        DT_3D_div,
        T_MAX_3D_div,
        output_every=OUTPUT_EVERY_3D_div,
        gamma2=GAMMA2_3D_div,
        rk_order=4,
    )

    # --- Chyba vůči exaktnímu řešení ---
    # Exaktní řešení je pravoběžný gaussián na celé přímce, takže v naší domény platí
    # jen dokud puls nedorazí na vnější okraj -- to je exit_time(run_3d_div).
    error_3d_div = run_3d_div.psi - exact_psi_3d(run_3d_div)
    error_l2_3d_div = l2_norm_history(error_3d_div, run_3d_div.dr)

    # --- Porušení constraintu C = D6 psi - phi ---
    # S phi_source="discrete" (default) je C(0) = 0 identicky, takže všechno naměřené
    # vzniklo z okrajových podmínek. Na test tlumení C(t) = C(0) exp(-gamma2 t) je
    # potřeba běh s C(0) != 0, tj. phi_source="analytic" -- viz verify.py, test C.
    constraint_l2_3d_div = l2_norm_history(
        constraint_violation(run_3d_div.psi, run_3d_div.phi, run_3d_div.dr), run_3d_div.dr
    )

    t_exit_3d_div = exit_time(run_3d_div)
    before_3d_div = run_3d_div.t <= t_exit_3d_div

    print(f"DR = {DR_3D_div}:")
    print(f"exit_time         = {t_exit_3d_div:.2f}")
    print(f"||e||_2 max do    = {error_l2_3d_div[before_3d_div].max():.3e}")
    print(
        f"||C||_2           = {constraint_l2_3d_div[0]:.3e} -> {constraint_l2_3d_div[before_3d_div][-1]:.3e}"
    )
    print(
        f"||e||_2 max po    = {error_l2_3d_div[~before_3d_div].max():.3e}  (odraz od okraje, ne chyba)"
    )
    print()
    # Za exit_time je exaktní řešení v doméně nulové, takže ||e|| = ||psi|| a měří jen
    # to, co po pulsu zbylo -- odraz od Sommerfeldovy podmínky. Konverguje k nule
    # s dr^5 (řád vnějšího okraje), takže při dr <= 0.05 je pod zaokrouhlením.

    # --- Grafy ---
    # Use the specific subplots for the current DR value
    ax1 = axes_all[i, 0]
    ax2 = axes_all[i, 1]

    ax1.semilogy(run_3d_div.t, error_l2_3d_div, lw=1.5, label=r"$\|e\|_2$")
    ax1.semilogy(run_3d_div.t, constraint_l2_3d_div, lw=1.5, label=r"$\|C\|_2$")
    ax1.axvline(exit_time(run_3d_div), color="black", lw=1, ls=":", label="odchod pulsu")
    ax1.set_xlabel("t")
    ax1.set_title(f"L2 normy (DR = {run_3d_div.dr:g})")
    ax1.grid(True, ls="--", alpha=0.6)
    ax1.legend()

    scale_3d_div = abs(error_3d_div).max() or 1.0
    mesh_3d_div = ax2.pcolormesh(
        run_3d_div.r,
        run_3d_div.t,
        error_3d_div,
        cmap="RdBu_r",
        vmin=-scale_3d_div,
        vmax=scale_3d_div,
        shading="nearest",
    )
    fig_all.colorbar(mesh_3d_div, ax=ax2, label="e")
    ax2.set_xlabel("r")
    ax2.set_ylabel("t")
    ax2.set_title(f"e(t, r) (DR = {run_3d_div.dr:g})")

# Adjust layout and display all plots
fig_all.tight_layout()
plt.show()

## 3D případ, rozepsaný tvar

### Utils

In [ ]:
def run_3d_gaussian(
    dr: float,
    dt: float,
    tmax: float,
    *,
    rmax: float = 10.0,
    r0: float = 4.0,
    sigma: float = 0.5,
    gamma2: float = 0.1,
    rk_order: int = 4,
    output_every: int = 1,
) -> Run:
    """Odintegruje jeden běh a vrátí ho jako `Run`.

    Args:
        dr, dt: prostorový a časový krok.
        tmax: koncový čas, zaokrouhlí se na nejbližší násobek dt.
        dimension_n: počet prostorových rozměrů (3 = sférická symetrie).
        rhs_builder: `ndim_lin_wave_equation_rhs` (nedivergenční) nebo
            `ndim_lin_wave_equation_div_rhs` (divergenční) z `we_systems`.
        output_every: ukládá se každý output_every-tý krok. Při porovnávání
            rozlišení ho škáluj s dt, aby běhy měly stejné výstupní časy.
    """
    r = grid(dr, rmax)
    n_points = len(r)
    psi, phi, pi = initial_data_3d(r, r0, sigma)

    solver = RKp(order=rk_order)
    solver.Initialize(
        np.concatenate([psi, phi, pi]),
        ndim_lin_wave_equation_rhs(
            r_coordinate=r,
            dimension_n=DIMENSION_3,
            dx=dr,
            gamma2=gamma2,
            N_of_points=n_points,
        ),
    )
    solver.Integrate(t0=0.0, h=dt, tmax=tmax, output_every=output_every)

    return Run.from_solver(
        solver,
        r=r,
        meta={
            "r0": r0,
            "sigma": sigma,
            "gamma2": gamma2,
            "rk_order": rk_order,
            "dt_integration": dt,
            "dimension_n": DIMENSION_3,
            "system": ndim_lin_wave_equation_rhs.__name__,
        },
    )

### Analýza

In [ ]:
import matplotlib.pyplot as plt

# --- Parametry ---
DR_3D = 0.1
DT_3D = 0.005
T_MAX_3D = 8.0
OUTPUT_EVERY_3D = 10
GAMMA2_3D = 0.1

assert cfl_check(DT_3D, DR_3D), "CFL violated"

run_3d = run_3d_gaussian(
    DR_3D, DT_3D, T_MAX_3D, output_every=OUTPUT_EVERY_3D, gamma2=GAMMA2_3D, rk_order=4
)

# --- Chyba vůči exaktnímu řešení ---
# Exaktní řešení je pravoběžný gaussián na celé přímce, takže v naší domény platí
# jen dokud puls nedorazí na vnější okraj -- to je exit_time(run_3d).
error_3d = run_3d.psi - exact_psi_3d(run_3d)
error_l2_3d = l2_norm_history(error_3d, run_3d.dr)

# --- Porušení constraintu C = D6 psi - phi ---
# S phi_source="discrete" (default) je C(0) = 0 identicky, takže všechno naměřené
# vzniklo z okrajových podmínek. Na test tlumení C(t) = C(0) exp(-gamma2 t) je
# potřeba běh s C(0) != 0, tj. phi_source="analytic" -- viz verify.py, test C.
constraint_l2_3d = l2_norm_history(
    constraint_violation(run_3d.psi, run_3d.phi, run_3d.dr), run_3d.dr
)

t_exit_3d = exit_time(run_3d)
before_3d = run_3d.t <= t_exit_3d

print(f"exit_time         = {t_exit_3d:.2f}")
print(f"||e||_2 max do    = {error_l2_3d[before_3d].max():.3e}")
print(f"||C||_2           = {constraint_l2_3d[0]:.3e} -> {constraint_l2_3d[before_3d][-1]:.3e}")
print(f"||e||_2 max po    = {error_l2_3d[~before_3d].max():.3e}  (odraz od okraje, ne chyba)")
print()
# Za exit_time je exaktní řešení v doméně nulové, takže ||e|| = ||psi|| a měří jen
# to, co po pulsu zbylo -- odraz od Sommerfeldovy podmínky. Konverguje k nule
# s dr^5 (řád vnějšího okraje), takže při dr <= 0.05 je pod zaokrouhlením.

# --- Grafy ---
fig_3d, (ax1_3d, ax2_3d) = plt.subplots(1, 2, figsize=(13, 5))

ax1_3d.semilogy(run_3d.t, error_l2_3d, lw=1.5, label=r"$\|e\|_2$")
ax1_3d.semilogy(run_3d.t, constraint_l2_3d, lw=1.5, label=r"$\|C\|_2$")
ax1_3d.axvline(exit_time(run_3d), color="black", lw=1, ls=":", label="odchod pulsu")
ax1_3d.set_xlabel("t")
ax1_3d.set_title(f"L2 normy, dr = {run_3d.dr:g}")
ax1_3d.grid(True, ls="--", alpha=0.6)
ax1_3d.legend()

scale_3d = abs(error_3d).max() or 1.0
mesh_3d = ax2_3d.pcolormesh(
    run_3d.r, run_3d.t, error_3d, cmap="RdBu_r", vmin=-scale_3d, vmax=scale_3d, shading="nearest"
)
fig_3d.colorbar(mesh_3d, ax=ax2_3d, label="e")
ax2_3d.set_xlabel("r")
ax2_3d.set_ylabel("t")
ax2_3d.set_title(f"e(t, r), dr = {run_3d.dr:g}")

fig_3d.tight_layout()
plt.show()

### Řád konvergence

In [ ]:
T_MAX_3D = 4.0
GAMMA2_3D = 0.1
RK_ORDER_3D = 4

# ---------------------------------------------------------------------------
# Sken 1: prostorový řád proti exaktnímu řešení
# ---------------------------------------------------------------------------

DT_FIXED_3D = 0.0025  # tak malé, aby časová chyba RK4 byla zanedbatelná
DR_FIXED_3D_SCAN1 = 0.1  # Placeholder for the first scan's DR, if needed for CFL check
assert cfl_check(DT_FIXED_3D, DR_FIXED_3D_SCAN1), "CFL violated for Scan 1 (dt_fixed_3d, first dr)"

rows = []
prev = None
for dr in [0.1, 0.05, 0.025]:
    run_3d = run_3d_gaussian(
        dr, DT_FIXED_3D, T_MAX_3D, output_every=1, gamma2=GAMMA2_3D, rk_order=RK_ORDER_3D
    )
    err = l2_norm(run_3d.psi[-1] - exact_psi_3d(run_3d)[-1], run_3d.dr)
    rows.append(
        {
            "dr": dr,
            "err_l2": err,
            "ratio": prev / err if prev else np.nan,
            "order": np.log2(prev / err) if prev else np.nan,
        }
    )
    prev = err

print_table(rows, f"Sken 1 -- prostorový řád (dt_3d = {DT_FIXED_3D}, t_3d = {T_MAX_3D})   čekám 6")

# ---------------------------------------------------------------------------
# Sken 2: časový řád self-konvergencí
# ---------------------------------------------------------------------------

DR_FIXED_3D = 0.05
DT_COARSE_3D = 0.02  # dt/dr = 0.4, CFL v pohodě
assert cfl_check(DT_COARSE_3D, DR_FIXED_3D), "CFL violated for Scan 2 (dt_coarse_3d, dr_fixed_3d)"

psis_3d = []
for k in range(3):
    # output_every se zdvojuje s půlením dt -> snímky padnou na stejné časy
    run_3d = run_3d_gaussian(
        DR_FIXED_3D,
        DT_COARSE_3D / 2**k,
        T_MAX_3D,
        output_every=2**k,
        gamma2=GAMMA2_3D,
        rk_order=RK_ORDER_3D,
    )
    psis_3d.append(run_3d.psi[-1])

# space_factor=1: mřížka se nemění, zjemňuje se jen čas
order, d1, d2 = self_convergence(psis_3d[0], psis_3d[1], psis_3d[2], DR_FIXED_3D, space_factor=1)
print(
    f"\nSken 2 -- časový řád self-konvergencí (dr_3d = {DR_FIXED_3D}, t_3d = {T_MAX_3D})   čekám 4"
)
print(f"  ||u_dt - u_dt/2||   = {d1:.4e}")
print(f"  ||u_dt/2 - u_dt/4|| = {d2:.4e}")
print(f"  poměr               = {d1 / d2:.2f}   (má být 2^4 = 16)")
print(f"  řád                 = {order:.3f}")

### Iterace přes dR

In [ ]:
import matplotlib.pyplot as plt


def cfl_check(dt: float, dr: float, c: float = 1.0):
    return c * dt / dr <= 1


# --- Parametry ---
DT_3D = 0.005
T_MAX_3D = 16.0
OUTPUT_EVERY_3D = 10
GAMMA2_3D = 0.1

DR_values = [0.1, 0.05, 0.01, 0.005]
num_dr_values = len(DR_values)

# Create a single figure with multiple subplots
# Each DR will occupy one row with two subplots (L2 norms and e(t,r))
fig_all, axes_all = plt.subplots(num_dr_values, 2, figsize=(13, 5 * num_dr_values), squeeze=False)

for i, DR_3D in enumerate(DR_values):
    assert cfl_check(DT_3D, DR_3D), "CFL violated"

    run_3d = run_3d_gaussian(
        DR_3D, DT_3D, T_MAX_3D, output_every=OUTPUT_EVERY_3D, gamma2=GAMMA2_3D, rk_order=4
    )

    # --- Chyba vůči exaktnímu řešení ---
    error_3d = run_3d.psi - exact_psi_3d(run_3d)
    error_l2_3d = l2_norm_history(error_3d, run_3d.dr)

    # --- Logaritmus chyby ---
    error_3d = np.log10(np.abs(error_3d))

    # --- Klipneme nekonečna na maxima a minima * 1.2 (bude vidět skok ale zachová se rozumná škála) ---
    error_3d_noninf_max = error_3d[error_3d != np.inf].max() * 1.2
    error_3d_noninf_min = error_3d[error_3d != -np.inf].min() * 1.2

    error_3d = error_3d.clip(min=error_3d_noninf_min, max=error_3d_noninf_max)

    # --- Porušení constraintu C = D6 psi - phi ---
    constraint_l2_3d = l2_norm_history(
        constraint_violation(run_3d.psi, run_3d.phi, run_3d.dr), run_3d.dr
    )

    t_exit_3d = exit_time(run_3d)
    before_3d = run_3d.t <= t_exit_3d

    print(f"DR = {DR_3D}:")
    print(f"exit_time         = {t_exit_3d:.2f}")
    print(f"||e||_2 max do    = {error_l2_3d[before_3d].max():.3e}")
    print(f"||C||_2           = {constraint_l2_3d[0]:.3e} -> {constraint_l2_3d[before_3d][-1]:.3e}")
    print(f"||e||_2 max po    = {error_l2_3d[~before_3d].max():.3e}  (odraz od okraje, ne chyba)")
    print()

    # --- Grafy ---
    ax1 = axes_all[i, 0]
    ax2 = axes_all[i, 1]

    ax1.semilogy(run_3d.t, error_l2_3d, lw=1.5, label=r"$\|e\|_2$")
    ax1.semilogy(run_3d.t, constraint_l2_3d, lw=1.5, label=r"$\|C\|_2$")
    ax1.axvline(exit_time(run_3d), color="black", lw=1, ls=":", label="odchod pulsu")
    ax1.set_xlabel("t")
    ax1.set_title(f"L2 normy (nedivergentní, DR = {run_3d.dr:g})")
    ax1.grid(True, ls="--", alpha=0.6)
    ax1.legend()

    mesh_3d = ax2.pcolormesh(
        run_3d.r,
        run_3d.t,
        error_3d,
        cmap="RdBu_r",
        vmin=error_3d_noninf_min,
        vmax=error_3d_noninf_max,
        shading="nearest",
    )
    fig_all.colorbar(mesh_3d, ax=ax2, label="e")
    ax2.set_xlabel("r")
    ax2.set_ylabel("t")
    ax2.set_title(f"e(t, r) (nedivergentní, DR = {run_3d.dr:g})")

# Adjust layout and display all plots
fig_all.tight_layout()
plt.show()

## Animace

### Animace 1D případu

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Parametry pro 1D animaci ---
DR_1D = 0.01
DT_1D = 0.005
T_MAX_1D = 30.0  # Použijte delší T_MAX pro lepší animaci
OUTPUT_EVERY_1D = 20
GAMMA2_1D = 0.1
RK_ORDER_1D = 4

assert cfl_check(DT_1D, DR_1D), "CFL violated"

# Spusťte 1D gaussovskou simulaci
run_1d = run_1d_gaussian(
    DR_1D, DT_1D, T_MAX_1D, output_every=OUTPUT_EVERY_1D, gamma2=GAMMA2_1D, rk_order=RK_ORDER_1D
)

r_coord_1d = run_1d.r
psi_simulated_1d = run_1d.psi

r_min_1d = r_coord_1d.min()
r_max_1d = r_coord_1d.max()

# --- ANIMACE ---
every_nth_frame_1d = (
    5  # Zobrazit každý snímek z výstupu (který je již podvzorkován pomocí output_every)
)

fig_1d, ax_1d = plt.subplots(figsize=(10, 5))
(line_1d,) = ax_1d.plot(r_coord_1d, psi_simulated_1d[0, :], lw=2, color="firebrick")

ax_1d.set_xlim(r_min_1d, r_max_1d)
ax_1d.set_ylim(-1, 1)  # Předpokládáme, že hodnoty psi jsou v tomto rozsahu
ax_1d.set_title("Animace výsledků (1D)")
ax_1d.set_xlabel("Prostor (r)")
ax_1d.set_ylabel("Amplituda (psi)")
ax_1d.grid(True, linestyle="--", alpha=0.6)


def update_1d(frame):
    line_1d.set_ydata(psi_simulated_1d[frame, :])
    return (line_1d,)


ani_1d = FuncAnimation(
    fig_1d,
    update_1d,
    frames=len(psi_simulated_1d),
    interval=20
    * (
        OUTPUT_EVERY_1D // 10
    ),  # Upravte interval na základě původního output_every pro plynulejší animaci
    blit=True,
    cache_frame_data=False,
)

plt.close(fig_1d)  # Zavřete graf, aby se nezobrazoval dvakrát
HTML(ani_1d.to_jshtml())

### Animace 3D případu (divergenční tvar)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Parametry pro 3D divergentní animaci ---
DR_3D_div = 0.01
DT_3D_div = 0.0005
T_MAX_3D_div = 30.0  # Použijte delší T_MAX pro lepší animaci
OUTPUT_EVERY_3D_div = 1000
GAMMA2_3D_div = 0.1
RK_ORDER_3D_div = 4

assert cfl_check(DT_3D_div, DR_3D_div), "CFL violated"

# Spusťte 3D gaussovskou simulaci
run_3d_div = run_3d_gaussian_div(
    DR_3D_div,
    DT_3D_div,
    T_MAX_3D_div,
    output_every=OUTPUT_EVERY_3D_div,
    gamma2=GAMMA2_3D_div,
    rk_order=RK_ORDER_3D_div,
)

r_coord_3d_div = run_3d_div.r
psi_simulated_3d_div = run_3d_div.psi

r_min_3d_div = r_coord_3d_div.min()
r_max_3d_div = r_coord_3d_div.max()

# --- ANIMACE ---
every_nth_frame_3d_div = (
    10  # Zobrazit každý snímek z výstupu (který je již podvzorkován pomocí output_every)
)

fig_3d_div, ax_3d_div = plt.subplots(figsize=(10, 5))
(line_3d_div,) = ax_3d_div.plot(r_coord_3d_div, psi_simulated_3d_div[0, :], lw=2, color="firebrick")

ax_3d_div.set_xlim(r_min_3d_div, r_max_3d_div)
ax_3d_div.set_ylim(-1, 1)  # Předpokládáme, že hodnoty psi jsou v tomto rozsahu
ax_3d_div.set_title("Animace výsledků (3D divergentní)")
ax_3d_div.set_xlabel("Prostor (r)")
ax_3d_div.set_ylabel("Amplituda (psi)")
ax_3d_div.grid(True, linestyle="--", alpha=0.6)


def update_3d_div(frame):
    line_3d_div.set_ydata(psi_simulated_3d_div[frame, :])
    return (line_3d_div,)


ani_3d_div = FuncAnimation(
    fig_3d_div,
    update_3d_div,
    frames=len(psi_simulated_3d_div),
    interval=20
    * (
        OUTPUT_EVERY_3D_div // 10
    ),  # Upravte interval na základě původního output_every pro plynulejší animaci
    blit=True,
    cache_frame_data=False,
)

plt.close(fig_3d_div)  # Zavřete graf, aby se nezobrazoval dvakrát
HTML(ani_3d_div.to_jshtml())

### Animace 3D případu (nedivergenční tvar)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Parametry pro 3D animaci ---
DR_3D = 0.1
DT_3D = 0.005
T_MAX_3D = 8.0  # Použijte delší T_MAX pro lepší animaci
OUTPUT_EVERY_3D = 10
GAMMA2_3D = 0.1
RK_ORDER_3D = 4

# Spusťte 3D gaussovskou simulaci (nedivergenční)
run_3d = run_3d_gaussian(
    DR_3D, DT_3D, T_MAX_3D, output_every=OUTPUT_EVERY_3D, gamma2=GAMMA2_3D, rk_order=RK_ORDER_3D
)

r_coord_3d = run_3d.r
psi_simulated_3d = run_3d.psi

r_min_3d = r_coord_3d.min()
r_max_3d = r_coord_3d.max()

# --- ANIMACE ---
every_nth_frame_3d = (
    1  # Zobrazit každý snímek z výstupu (který je již podvzorkován pomocí output_every)
)

fig_3d, ax_3d = plt.subplots(figsize=(10, 5))
(line_3d,) = ax_3d.plot(r_coord_3d, psi_simulated_3d[0, :], lw=2, color="firebrick")

ax_3d.set_xlim(r_min_3d, r_max_3d)
ax_3d.set_ylim(-1, 1)  # Předpokládáme, že hodnoty psi jsou v tomto rozsahu
ax_3d.set_title("Animace výsledků (3D)")
ax_3d.set_xlabel("Prostor (r)")
ax_3d.set_ylabel("Amplituda (psi)")
ax_3d.grid(True, linestyle="--", alpha=0.6)


def update_3d(frame):
    line_3d.set_ydata(psi_simulated_3d[frame, :])
    return (line_3d,)


ani_3d = FuncAnimation(
    fig_3d,
    update_3d,
    frames=len(psi_simulated_3d),
    interval=20
    * (
        OUTPUT_EVERY_3D // 10
    ),  # Upravte interval na základě původního output_every pro plynulejší animaci
    blit=True,
    cache_frame_data=False,
)

plt.close(fig_3d)  # Zavřete graf, aby se nezobrazoval dvakrát
HTML(ani_3d.to_jshtml())

### Animace 3D num divergenční tvar - 3D exact

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np


def cfl_check(dt: float, dr: float, c: float = 1.0):
    return c * dt / dr <= 1


# --- Parametry pro 3D divergentní animaci chyby ---
DR_3D_div_error = 0.01
DT_3D_div_error = 0.0005
T_MAX_3D_div_error = 30.0
OUTPUT_EVERY_3D_div_error = 1000
GAMMA2_3D_div_error = 0.1
RK_ORDER_3D_div_error = 4

assert cfl_check(DT_3D_div_error, DR_3D_div_error), "CFL violated for 3D divergent error animation"

# Spustíme 3D běh v divergenčním tvaru
run_3d_div_error = run_3d_gaussian_div(
    DR_3D_div_error,
    DT_3D_div_error,
    T_MAX_3D_div_error,
    output_every=OUTPUT_EVERY_3D_div_error,
    gamma2=GAMMA2_3D_div_error,
    rk_order=RK_ORDER_3D_div_error,
)

# Vypočítáme exaktní řešení
exact_psi_3d_val_error = exact_psi_3d(run_3d_div_error)

# Vypočítáme chybu (numerické řešení - exaktní řešení)
error_3d_div_anim = run_3d_div_error.psi - exact_psi_3d_val_error

r_coord_3d_div_error = run_3d_div_error.r

r_min_3d_div_error = r_coord_3d_div_error.min()
r_max_3d_div_error = r_coord_3d_div_error.max()

# --- Grafické zobrazení a animace ---
fig_3d_div_error, ax_3d_div_error = plt.subplots(figsize=(10, 5))
(line_3d_div_error,) = ax_3d_div_error.plot(
    r_coord_3d_div_error, error_3d_div_anim[0, :], lw=2, color="blue"
)

ax_3d_div_error.set_xlim(r_min_3d_div_error, r_max_3d_div_error)
ax_3d_div_error.set_ylim(-error_3d_div_anim.max() * 1.1, error_3d_div_anim.max() * 1.1)
ax_3d_div_error.set_title("Animace chyby (3D divergentní num_sol - exact_sol)")
ax_3d_div_error.set_xlabel("Prostor (r)")
ax_3d_div_error.set_ylabel("Chyba (e)")
ax_3d_div_error.grid(True, linestyle="--", alpha=0.6)


def update_3d_div_error(frame):
    line_3d_div_error.set_ydata(error_3d_div_anim[frame, :])
    return (line_3d_div_error,)


ani_3d_div_error = FuncAnimation(
    fig_3d_div_error,
    update_3d_div_error,
    frames=len(error_3d_div_anim),
    interval=20 * (OUTPUT_EVERY_3D_div_error // 100),
    blit=True,
    cache_frame_data=False,
)

plt.close(fig_3d_div_error)
HTML(ani_3d_div_error.to_jshtml())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- ANIMACE ---
fig_3d_div_error, ax_3d_div_error = plt.subplots(figsize=(10, 5))
(line_3d_div_error,) = ax_3d_div_error.plot(
    r_coord_3d_div_error, error_3d_div_anim[0, :], lw=2, color="blue"
)

ax_3d_div_error.set_xlim(r_min_3d_div_error, r_max_3d_div_error)
ax_3d_div_error.set_ylim(
    -error_3d_div_anim.max() * 1.1, error_3d_div_anim.max() * 1.1
)  # Dynamické omezení rozsahu y
ax_3d_div_error.set_title("Animace chyby (3D divergentní num_sol - exact_sol)")
ax_3d_div_error.set_xlabel("Prostor (r)")
ax_3d_div_error.set_ylabel("Chyba (e)")
ax_3d_div_error.grid(True, linestyle="--", alpha=0.6)


def update_3d_div_error(frame):
    line_3d_div_error.set_ydata(error_3d_div_anim[frame, :])
    return (line_3d_div_error,)


ani_3d_div_error = FuncAnimation(
    fig_3d_div_error,
    update_3d_div_error,
    frames=len(error_3d_div_anim),
    interval=20 * (OUTPUT_EVERY_3D_div_error // 100),  # Upravte interval pro plynulejší animaci
    blit=True,
    cache_frame_data=False,
)

plt.close(fig_3d_div_error)  # Zavřete graf, aby se nezobrazoval dvakrát
HTML(ani_3d_div_error.to_jshtml())

### Animace 3D num rozepsaný tvar - 3D exact

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Parametry pro 3D nedivergentní animaci chyby ---
DR_3D_error = 0.01
DT_3D_error = 0.0005
T_MAX_3D_error = 30.0
OUTPUT_EVERY_3D_error = 1000
GAMMA2_3D_error = 0.1
RK_ORDER_3D_error = 4

assert cfl_check(DT_3D_error, DR_3D_error), "CFL violated for 3D non-divergent error animation"

# Spustíme 3D běh v nedivergenčním tvaru
run_3d_error = run_3d_gaussian(
    DR_3D_error,
    DT_3D_error,
    T_MAX_3D_error,
    output_every=OUTPUT_EVERY_3D_error,
    gamma2=GAMMA2_3D_error,
    rk_order=RK_ORDER_3D_error,
)

# Vypočítáme exaktní řešení
exact_psi_3d_val_error_non_div = exact_psi_3d(run_3d_error)

# Vypočítáme chybu (numerické řešení - exaktní řešení)
error_3d_non_div_anim = run_3d_error.psi - exact_psi_3d_val_error_non_div

r_coord_3d_error = run_3d_error.r

r_min_3d_error = r_coord_3d_error.min()
r_max_3d_error = r_coord_3d_error.max()

# --- Grafické zobrazení a animace ---
fig_3d_error, ax_3d_error = plt.subplots(figsize=(10, 5))
(line_3d_error,) = ax_3d_error.plot(
    r_coord_3d_error, error_3d_non_div_anim[0, :], lw=2, color="blue"
)

ax_3d_error.set_xlim(r_min_3d_error, r_max_3d_error)
ax_3d_error.set_ylim(-error_3d_non_div_anim.max() * 1.1, error_3d_non_div_anim.max() * 1.1)
ax_3d_error.set_title("Animace chyby (3D nedivergentní num_sol - exact_sol)")
ax_3d_error.set_xlabel("Prostor (r)")
ax_3d_error.set_ylabel("Chyba (e)")
ax_3d_error.grid(True, linestyle="--", alpha=0.6)


def update_3d_error(frame):
    line_3d_error.set_ydata(error_3d_non_div_anim[frame, :])
    return (line_3d_error,)


ani_3d_error = FuncAnimation(
    fig_3d_error,
    update_3d_error,
    frames=len(error_3d_non_div_anim),
    interval=20 * (OUTPUT_EVERY_3D_error // 100),
    blit=True,
    cache_frame_data=False,
)

plt.close(fig_3d_error)
HTML(ani_3d_error.to_jshtml())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- ANIMACE ---
fig_3d_error, ax_3d_error = plt.subplots(figsize=(10, 5))
(line_3d_error,) = ax_3d_error.plot(
    r_coord_3d_error, error_3d_non_div_anim[0, :], lw=2, color="blue"
)

ax_3d_error.set_xlim(r_min_3d_error, r_max_3d_error)
ax_3d_error.set_ylim(-1e-2, 1e-2)  # Chyba v jednotkách řešení
ax_3d_error.set_title("Animace chyby (3D nedivergentní num_sol - exact_sol)")
ax_3d_error.set_xlabel("Prostor (r)")
ax_3d_error.set_ylabel("Chyba (e)")
ax_3d_error.grid(True, linestyle="--", alpha=0.6)


def update_3d_error(frame):
    line_3d_error.set_ydata(error_3d_non_div_anim[frame, :])
    return (line_3d_error,)


ani_3d_error = FuncAnimation(
    fig_3d_error,
    update_3d_error,
    frames=len(error_3d_non_div_anim),
    interval=20 * (OUTPUT_EVERY_3D_error // 100),  # Upravte interval pro plynulejší animaci
    blit=True,
    cache_frame_data=False,
)

plt.close(fig_3d_error)  # Zavřete graf, aby se nezobrazoval dvakrát
HTML(ani_3d_error.to_jshtml())

# Popis problému

V sekcích 3D je podsekce "Iterace před dR".
Tato podsekce v obou sekcích představuje analýzu chování při zmenšování dR za konstantního dT, přičemž dT je voleno tak, aby byla zachována CFL podmínka.

Chyba u divergenčního tvaru se chová inteligentně. Snižuje se postupně.

Chyba u nedivergenčního tvaru se chová inteligentně jen do dR = 0.05, pak vybouchne.

Nevím proč. Podezřívám, že 1/R bude mít rostoucí mód v počátku, protože není u operátoru derivace.